## Ciclos Solares:

- __Ciclo Solar 23__: Máximo em março de 2000 (número máximo suavizado de manchas solares: 120.8)
- __Ciclo Solar 24__: Dois picos, em 2011 (99) e início de 2014 (101)
- __Ciclo Solar 25__: Previsto entre 2023 e 2026, com um intervalo de manchas solares de 95 a 130, marcando o atual período de máximo solar em 2026

# 2. Earth Quakes - USGS

In [ ]:
# 2.1. Instalações:

%pip install spaceweather -q
%pip install spacepy -q

# 2.2. Importações
import pandas as pd
import requests
import time

# 2.3. Requisição API da USGS
print("Buscando dados de terremotos no USGS...")

# Definir parâmetros de busca (Ex: Ano de 2023 completo)
filtros ={
  "format": "geojson",
  "starttime": "2002-01-01T00:00:00", # 2026-01-01T14:00:00
  "endtime": "2004-12-31T23:59:59",
  "minmagnitude": "4.0"  # Filtro para terremotos moderados a fortes
}

features_total =[]

for ano in range(2008, 2014):
  filtros['starttime'] = f"{ano}-01-01T00:00:00"
  filtros['endtime'] = f"{ano}-12-31T23:59:59"

  try:
    print(f' - extraindo {ano}: ', end=' ')
    response =requests.get(usgs_url, params=filtros)
    print(response.status_code)
    usgs_data =response.json()
    usgs_fea =usgs_data.get('features', [])
    features_total.extend(usgs_fea)
    time.sleep(3)
  except Exception as e:
    print(f"Erro ao acessar {ano}: {e}")

# 2.4. conversão de Data API para Dataframe
df_EQ =pd.json_normalize(features_total)
print(df_EQ.shape)

# 2.5. seleção de colunas
selectedColumns =[
  'properties.mag',
  'properties.place',
  'properties.time',
  'geometry.coordinates'
]

df_EQ =df_EQ[selectedColumns].rename(columns={
    'properties.mag': 'magnitude',
    'properties.place': 'local',
    'properties.time': 'UTC',
    'geometry.coordinates': 'coordenadas'
})

# 2.6. Indexagem da data
df_EQ =df_EQ.set_index('UTC')
df_EQ =df_EQ.sort_index()

# 2.7. [Earth-Quakes] conversão de tipos
df_EQ.index =pd.to_datetime(df_EQ.index, unit='ms')

print("Total: ", df_EQ.shape)
print("\nApartir de 5.0: ", df_EQ[df_EQ['magnitude'] >= 5.0].shape[0])
print("\nOrdenado: \n", df_EQ[df_EQ['magnitude'] >= 5.0].sort_values(by='magnitude', ascending=False).head())
print(df_EQ[(df_EQ['local'].str.contains('Philippines', na=False)) & (df_EQ['magnitude'] >= 5.0)].sort_values(by='magnitude', ascending=False).head())

# 2.8. Plotagem:
import matplotlib.pyplot as plt

# dataRange =['2000-01-01', '2010-12-31']

eq_byday = df_EQ.groupby(df_EQ.index.date).agg(
    maxMag=('magnitude', 'max'),
    eqQnt=('magnitude', 'count')
)

print(df_EQ.shape)
print(eq_byday.shape)
print(eq_byday.loc[[eq_byday['eqQnt'].idxmax()]]) # linha do dia onde houve maior incidencias
print(df_EQ.loc['2011-03-11'])

# 2.8.2. Converter o índice de volta para datetime para o matplotlib formatar o eixo X corretamente
eq_byday1 = eq_byday
eq_byday1.index = pd.to_datetime(eq_byday.index)

fig, ax1 = plt.subplots(figsize=(12, 6))

color ='tab:red'
ax1.set_xlabel('Data')
ax1.set_ylabel('Maior Magnitude do Dia', color=color)
ax1.plot(eq_byday1.index, eq_byday1['maxMag'], color=color, linewidth=0.8, label='Maior Magnitude')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

# 3. Criar o segundo eixo compartilhando o mesmo eixo X (Eixo Direito - Quantidade)
ax2 = ax1.twinx()

color = 'tab:blue'
ax2.set_ylabel('Quantidade de Terremotos', color=color)
ax2.bar(eq_byday1.index, eq_byday1['eqQnt'], color=color, alpha=0.5, width=0.5, label='Frequência')
ax2.tick_params(axis='y', labelcolor=color)

# 4. Ajustes finais de exibição
plt.title('Atividade Sísmica Diária: Magnitude Máxima vs. Frequência')
fig.tight_layout()
plt.show()

# 2.9.1. plotagem 2

# Converter o índice de volta para datetime para o matplotlib formatar o eixo X corretamente
eq_byday1 = eq_byday
eq_byday1.index = pd.to_datetime(eq_byday.index)


fig, ax1 = plt.subplots(figsize=(12, 6))

color ='tab:red'
ax1.set_xlabel('Data')
ax1.set_ylabel('Maior Magnitude do Dia', color=color)
ax1.plot(eq_byday1.index, eq_byday1['maxMag'], color=color, linewidth=0.8, label='Maior Magnitude')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

# 3. Criar o segundo eixo compartilhando o mesmo eixo X (Eixo Direito - Quantidade)
ax2 = ax1.twinx()

color = 'tab:blue'
ax2.set_ylabel('Quantidade de Terremotos', color=color)
ax2.bar(eq_byday1.index, eq_byday1['eqQnt'], color=color, alpha=0.5, width=0.5, label='Frequência')
ax2.tick_params(axis='y', labelcolor=color)

# 4. Ajustes finais de exibição
plt.title('Atividade Sísmica Diária: Magnitude Máxima vs. Frequência')
fig.tight_layout()
plt.show()

# 1. Solar X-Rays Flux - GOES

In [ ]:
!pip install sunpy[all] -q
from sunpy.net import Fido, attrs as a
from sunpy.timeseries import TimeSeries

tempo = a.Time('2013-01-01T00:00:00', '2013-01-31T23:59:59')
instrumento = a.Instrument('XRS')
satelite = a.goes.SatelliteNumber(18)

# 1.3.1. Faz a requisição à API da NOAA através do Fido (SunPy)
# Buscando pelo instrumento XRS (X-ray Sensor) responsável pelo fluxo
resultado = Fido.search(tempo, instrumento, satelite)

# 1.3.2. Faz o download dos arquivos (eles serão salvos em cache local)
arquivos_baixados = Fido.fetch(resultado)

# 1.3.3. Carrega e concatena os arquivos em uma Série Temporal do SunPy
# O SunPy resolve automaticamente a junção de múltiplos arquivos/anos
ts_goes = TimeSeries(arquivos_baixados, concatenate=True)

# 1.3.4. Converte a série temporal diretamente para um Pandas DataFrame
dfXRay = ts_goes.to_dataframe()

# Solar X-Ray Flux - Kaggle

In [ ]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("rjp7777/space-weather-and-geomagnetic-parameters")
print("Path to dataset files:", path)

arquivo =os.path.join(path, "data_solar.csv")
df_geoMag =pd.read_csv(arquivo)

# 1. Definir o primeiro dia de cada ano (Year-01-01)
data_base = pd.to_datetime(df_geoMag['Year'].astype(str) + '-01-01')

# 2. Somar o DOY (subtraindo 1 dia, pois o dia 1 já está incluso em 01-01)
dias_para_somar = pd.to_timedelta(df_geoMag['DOY'] - 1, unit='D')

# 3. Somar as horas
horas_para_somar = pd.to_timedelta(df_geoMag['Hour'], unit='h')

# Resultado Final
df_geoMag['UTC'] = data_base + dias_para_somar + horas_para_somar

df_geoMag = df_geoMag.drop(columns=['Year', 'DOY', 'Hour'])

df_geoMag =df_geoMag.set_index('UTC')
df_geoMag =df_geoMag.sort_index()

print(df_geoMag.head())

# Solar - Escala S (radiacao solar - Protons, eletrons e ions pesados)

In [ ]:
# 2.3. [Escala S] Baixando dados do satelite

tempo = a.Time('2026-06-01T00:00:00', '2026-06-03T23:59:59')
instrumento = a.Instrument('seiss')
# satelite = a.goes.SatelliteNumber(17)

# 2.3.1. Faz a requisição à API da NOAA através do Fido (SunPy)
# Buscando pelo instrumento XRS (X-ray Sensor) responsável pelo fluxo
resultado = Fido.search(tempo, instrumento, satelite)

# 2.3.2. Faz o download dos arquivos (eles serão salvos em cache local)
arquivos_baixados = Fido.fetch(resultado)

# 2.3.3. Carrega e concatena os arquivos em uma Série Temporal do SunPy
# O SunPy resolve automaticamente a junção de múltiplos arquivos/anos
ts_goes = TimeSeries(arquivos_baixados, concatenate=True)

# 2.3.4. Converte a série temporal diretamente para um Pandas DataFrame
df_Rad = ts_goes.to_dataframe()

# Solar - Sunspots [source](https://www.sidc.be/SILSO/datafiles)

In [ ]:
import pandas as pd

# 1.2.1 Definir os nomes corretos das colunas
newcolumns = ['ano', 'mes', 'dia', 'data_decimal', 'num_manchas', 'desvio_padrao', 'num_observacoes', 'status']

# 1.2.2. baixar, corrigindo o separador e os espaços
df_Sunspots_1818 =pd.read_csv("https://www.sidc.be/SILSO/INFO/sndtotcsv.php", sep=';', names=newcolumns, skipinitialspace=True)

# 1.2.3. Modificar campos de ano, mes e dia para um campo tipo datetime:
df_Sunspots_1818.rename(columns={'data':'UTC'}, inplace=True)
df_Sunspots_1818['UTC'] =pd.to_datetime(df_Sunspots_1818[['ano', 'mes', 'dia']].rename(columns={'ano':'year', 'mes':'month', 'dia':'day'}))
df_Sunspots_1818 =df_Sunspots_1818.drop(columns=['ano', 'mes', 'dia'])

# 1.2.4. Indexando coluna da data ('UTC')
df_Sunspots_1818.set_index('UTC', inplace=True)

# 1.3. Tratamento dos dados:

# 1.3.1 Tratar os valores ausentes (-1) para que não apareçam no gráfico:
df_Sunspots_1818['num_manchas'] = df_Sunspots_1818['num_manchas'].replace(-1, None)
df_Sunspots_1818 =df_Sunspots_1818[df_Sunspots_1818['num_manchas'].notnull()]

# Vulcoes

In [ ]:
import pandas as pd

df_Etna =pd.read_csv("https://osf.io/download/vrxa4")
df_Klauea =pd.read_csv("https://osf.io/download/8fjqd")
df_Merapi =pd.read_csv("https://osf.io/download/4qhpj")

# 1.2 Tratamento dos dados:

df_Etna['UTC'] =pd.to_datetime(df_Etna['UTC'])
df_Klauea['UTC'] =pd.to_datetime(df_Klauea['UTC'])

df_Etna =df_Etna.set_index('UTC')
df_Klauea =df_Klauea.set_index('UTC')

df_Etna =df_Etna[df_Etna['VRP'].notna()]
df_Klauea =df_Klauea[df_Klauea['VRP'].notna()]

# 1.3 Janela Mensal:

# 1.3.1. Agrupa por mês extraindo a média do VRP e aplica a janela deslizante (ex: 12 meses)
# janela_meses = 12
# vrp_mensal = df_Etna.resample('ME').mean()

# 1.3.2. Agrupa por mês (Média do VRP total global detectado naquele mês)
# vrp_suavizado = vrp_mensal.rolling(window=janela_meses, center=True).mean()

# 1.3.1 agrupando para medias e maximos diarias
dfVrp_Etna_d_m = df_Etna.resample('D').mean().dropna()
dfVrp_Etna_d_M = df_Etna.resample('D').max().dropna()

# 1.4. Plotagem 1 [com fórmula Min-Max]
import matplotlib.pyplot as plt

dataRange =['2000-01-01', '2010-12-31']

# 1.4.1 Aplica a fórmula Min-Max para normalização da coluna 'num_manchas'
sunspots_min =df_Sunspots_1818['num_manchas'].min()
sunspots_max =df_Sunspots_1818['num_manchas'].max()
sunspots_norm =(df_Sunspots_1818['num_manchas'] -sunspots_min) /(sunspots_max -sunspots_min)
vrp_min =dfVrp_Etna_d_M['VRP'].min()
vrp_max =dfVrp_Etna_d_M['VRP'].max()
vrp_norm =(dfVrp_Etna_d_M['VRP'] -vrp_min) /(vrp_max -vrp_min)

# 1.4.2. Escolha de janela temporal:
df1 =df_Sunspots_1818.loc[dataRange[0] : dataRange[1]]
df1['num_manchas'] =sunspots_norm
df2 =dfVrp_Etna_d_M.loc[dataRange[0] : dataRange[1]]
df2['VRP'] =vrp_norm

# df2 =df_Klauea.loc[dataRange[0] : dataRange[1]]

plt.figure(figsize=(12, 5))
plt.plot(df1.index, df1['num_manchas'], label='Manchas Solares', color='lightcoral', alpha=0.5)
plt.plot(df2.index, df2['VRP'], label='Maximos de Etna', color='red', linewidth=2)
# plt.plot(df2.index, df2['VRP'], label='Klauea', color='blue', linewidth=1)

plt.title(f"Poder Radiativo Vulcânico (VRP) - Etna ")
plt.xlabel("Ano")
plt.ylabel("VRP (Watts)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Modelo: Rede Neural

- __Abordagem__: LSTM (Long Short-Term Memory)
- __Objetivo__: reter informações de longo prazo

In [ ]:
!pip install tensorflow scikit-learn 
%pip install numpy

# Importação: 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Normalização
scaler = MinMaxScaler(feature_range=(0, 1))
dados_normalizados = scaler.fit_transform(df_Sunspots_1818[['num_manchas']])

# 12 meses anteriores para prever o próximo mês
def criar_janelas(dados, janela=12):
  X, y = [], []
  for i in range(len(dados) - janela):
    X.append(dados[i:(i + janela), 0])
    y.append(dados[i + janela, 0])
  return np.array(X), np.array(y)

JANELA_TEMPORAL = 12 #365
X, y = criar_janelas(dados_normalizados, JANELA_TEMPORAL)

# entrada 3D: [Amostras, Passos de Tempo, Recursos]
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

# Separar em Treino (80%) e Teste (20%) - IMPORTANTE: Sem embaralhar (shuffle=False)
tamanho_treino = int(len(X) * 0.8)
X_treino, X_teste = X[:tamanho_treino], X[tamanho_treino:]
y_treino, y_teste = y[:tamanho_treino], y[tamanho_treino:]


modelo = Sequential([
    # Primeira camada LSTM. input_shape indica o tamanho da janela e 1 recurso
    LSTM(units=50, return_sequences=True, input_shape=(JANELA_TEMPORAL, 1)),
    Dropout(0.2), # Evita overfitting deletando neurônios aleatórios no treino
    
    # Segunda camada LSTM
    LSTM(units=50, return_sequences=False),
    Dropout(0.2),
    
    # Camada de Saída (Gera 1 número contínuo, que é a previsão)
    Dense(units=1)
])

# Compilar o modelo usando o otimizador Adam e erro médio quadrático (MSE)
modelo.compile(optimizer='adam', loss='mean_squared_error')

# Treinar o modelo
historico = modelo.fit(
    X_treino, y_treino, 
    epochs=30, #20       # Quantas vezes ele vai passar por todo o dataset
    batch_size=16, #256  # Quantas amostras ele processa antes de atualizar os pesos
    validation_data=(X_teste, y_teste),
    verbose=1
)

# previsao
previsoes_norm = modelo.predict(X_teste)

# Voltando os dados para a escala original de manchas solares
previsoes = scaler.inverse_transform(previsoes_norm)
y_teste_original = scaler.inverse_transform(y_teste.reshape(-1, 1))


plt.figure(figsize=(12, 6))
plt.plot(y_teste_original, label='Dados Reais (num_manchas)', color='blue', marker='o')
plt.plot(previsoes, label='Previsão da Rede Neural (LSTM)', color='red', linestyle='--', marker='x')
plt.title('Previsão de Manchas Solares com Deep Learning')
plt.xlabel('Meses do Período de Teste')
plt.ylabel('Número de Manchas')
plt.legend()
plt.show()